In [31]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, ConfusionMatrixDisplay)
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_score, StratifiedKFold
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.tools.tools import add_constant
import seaborn as sns

data_s=pd.read_csv('owid-energy-data.csv')
pd.set_option('display.max_info_columns', 200)

data_s.info()
np.random.seed(42)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 23377 entries, 0 to 23376
Data columns (total 130 columns):
 #    Column                                        Non-Null Count  Dtype  
---   ------                                        --------------  -----  
 0    country                                       23377 non-null  object 
 1    year                                          23377 non-null  int64  
 2    iso_code                                      17265 non-null  object 
 3    population                                    18894 non-null  float64
 4    gdp                                           11780 non-null  float64
 5    biofuel_cons_change_pct                       2070 non-null   float64
 6    biofuel_cons_change_twh                       7903 non-null   float64
 7    biofuel_cons_per_capita                       6338 non-null   float64
 8    biofuel_consumption                           8057 non-null   float64
 9    biofuel_elec_per_capita                       60

In [3]:
# Stage 1 of cleaning database - remove data for continents

country=pd.DataFrame(data_s[data_s["iso_code"].notnull()])
country.head(200)

,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
26,Afghanistan,1900,AFG,4707744.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,Afghanistan,1901,AFG,4751177.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,Afghanistan,1902,AFG,4802500.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29,Afghanistan,1903,AFG,4861833.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30,Afghanistan,1904,AFG,4921891.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
593,Albania,1970,ALB,2181226.0,6.888219e+09,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
594,Albania,1971,ALB,2234995.0,7.337364e+09,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
595,Albania,1972,ALB,2289265.0,7.811525e+09,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
596,Albania,1973,ALB,2344108.0,8.317288e+09,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
country["iso_code"].unique()

array(['AFG', 'ALB', 'DZA', 'ASM', 'AGO', 'ATA', 'ATG', 'ARG', 'ARM',
       'ABW', 'AUS', 'AUT', 'AZE', 'BHS', 'BHR', 'BGD', 'BRB', 'BLR',
       'BEL', 'BLZ', 'BEN', 'BMU', 'BTN', 'BOL', 'BIH', 'BWA', 'BRA',
       'VGB', 'BRN', 'BGR', 'BFA', 'BDI', 'KHM', 'CMR', 'CAN', 'CPV',
       'CYM', 'CAF', 'TCD', 'CHL', 'CHN', 'COL', 'COM', 'COG', 'COK',
       'CRI', 'CIV', 'HRV', 'CUB', 'CUW', 'CYP', 'CZE', 'COD', 'DNK',
       'DJI', 'DMA', 'DOM', 'TLS', 'ECU', 'EGY', 'SLV', 'GNQ', 'ERI',
       'EST', 'SWZ', 'ETH', 'FLK', 'FRO', 'FJI', 'FIN', 'FRA', 'GUF',
       'PYF', 'GAB', 'GMB', 'GEO', 'DEU', 'GHA', 'GIB', 'GRC', 'GRL',
       'GRD', 'GLP', 'GUM', 'GTM', 'GIN', 'GNB', 'GUY', 'HTI', 'HND',
       'HKG', 'HUN', 'ISL', 'IND', 'IDN', 'IRN', 'IRQ', 'IRL', 'ISR',
       'ITA', 'JAM', 'JPN', 'JOR', 'KAZ', 'KEN', 'KIR', 'KWT', 'KGZ',
       'LAO', 'LVA', 'LBN', 'LSO', 'LBR', 'LBY', 'LTU', 'LUX', 'MAC',
       'MDG', 'MWI', 'MYS', 'MDV', 'MLI', 'MLT', 'MTQ', 'MRT', 'MUS',
       'MEX', 'FSM',

In [5]:
country['country'].unique()

array(['Afghanistan', 'Albania', 'Algeria', 'American Samoa', 'Angola',
       'Antarctica', 'Antigua and Barbuda', 'Argentina', 'Armenia',
       'Aruba', 'Australia', 'Austria', 'Azerbaijan', 'Bahamas',
       'Bahrain', 'Bangladesh', 'Barbados', 'Belarus', 'Belgium',
       'Belize', 'Benin', 'Bermuda', 'Bhutan', 'Bolivia',
       'Bosnia and Herzegovina', 'Botswana', 'Brazil',
       'British Virgin Islands', 'Brunei', 'Bulgaria', 'Burkina Faso',
       'Burundi', 'Cambodia', 'Cameroon', 'Canada', 'Cape Verde',
       'Cayman Islands', 'Central African Republic', 'Chad', 'Chile',
       'China', 'Colombia', 'Comoros', 'Congo', 'Cook Islands',
       'Costa Rica', "Cote d'Ivoire", 'Croatia', 'Cuba', 'Curacao',
       'Cyprus', 'Czechia', 'Democratic Republic of Congo', 'Denmark',
       'Djibouti', 'Dominica', 'Dominican Republic', 'East Timor',
       'Ecuador', 'Egypt', 'El Salvador', 'Equatorial Guinea', 'Eritrea',
       'Estonia', 'Eswatini', 'Ethiopia', 'Falkland Islands',
   

In [6]:
territories_to_exclude = [
    'ATA',  # Antarctica
    'ASM', 'GUM', 'MNP', 'PRI', 'VIR',  # US territories
    'BMU', 'VGB', 'CYM', 'FLK', 'GIB', 'MSR', 'SHN', 'TCA',  # British territories
    'GUF', 'PYF', 'GLP', 'MTQ', 'NCL', 'REU', 'SPM',  # French territories
    'FRO', 'GRL', 'CUW', 'ANT', 'COK', 'NIU',  # Danish/Dutch/NZ territories
    'HKG', 'MAC', 'TWN', 'ESH', 'PSE'  # judgment calls you chose to exclude
]
sovereign_countries =country[~country['iso_code'].isin(territories_to_exclude)]

In [7]:

print(type(sovereign_countries))
print(sovereign_countries.shape)
sovereign_countries.head(100)


<class 'pandas.core.frame.DataFrame'>
(15649, 130)


,country,year,iso_code,population,gdp,biofuel_cons_change_pct,biofuel_cons_change_twh,biofuel_cons_per_capita,biofuel_consumption,biofuel_elec_per_capita,...,solar_share_elec,solar_share_energy,wind_cons_change_pct,wind_cons_change_twh,wind_consumption,wind_elec_per_capita,wind_electricity,wind_energy_per_capita,wind_share_elec,wind_share_energy
26,Afghanistan,1900,AFG,4707744.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,Afghanistan,1901,AFG,4751177.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,Afghanistan,1902,AFG,4802500.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
29,Afghanistan,1903,AFG,4861833.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
30,Afghanistan,1904,AFG,4921891.0,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
121,Afghanistan,1995,AFG,17065837.0,1.230753e+10,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
122,Afghanistan,1996,AFG,17763266.0,1.207013e+10,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
123,Afghanistan,1997,AFG,18452100.0,1.185075e+10,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
124,Afghanistan,1998,AFG,19159996.0,1.169217e+10,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [8]:
# For every year, how many countries actually have a non-null value for each column.

pd.set_option('display.max_rows', None)
sovereign_countries.groupby('year')[['gdp', 'renewables_share_energy', 'fossil_share_energy']].count()

,gdp,renewables_share_energy,fossil_share_energy
year,,,
1900,36,0,0
1901,32,0,0
1902,33,0,0
1903,33,0,0
1904,33,0,0
1905,33,0,0
1906,34,0,0
1907,33,0,0
1908,33,0,0


In [9]:
# The set of country codes that had non-null renewables data in 1990.
# The second line does the exact same thing, but for year 2010.
# The & operator between two Python sets computes their intersection — the elements present in both sets

countries_1990 = set(sovereign_countries[sovereign_countries.year==1990].dropna(subset=['renewables_share_energy'])['iso_code'])
countries_2010 = set(sovereign_countries[sovereign_countries.year==2010].dropna(subset=['renewables_share_energy'])['iso_code'])
print(len(countries_1990 & countries_2010))  # intersection

77


In [10]:
# Checking the availability of data for the time frame t0=1995, t1=2010, t2=2015

countries_t0 = set(sovereign_countries[sovereign_countries.year==1995].dropna(subset=['renewables_share_energy','fossil_share_energy'])['iso_code'])
countries_t1 = set(sovereign_countries[sovereign_countries.year==2010].dropna(subset=['renewables_share_energy','fossil_share_energy','gdp'])['iso_code'])
countries_t2 = set(sovereign_countries[sovereign_countries.year==2015].dropna(subset=['gdp'])['iso_code'])

survivors = countries_t0 & countries_t1 & countries_t2
print(len(survivors))

77


In [11]:
# Data size after extending the set of features our model will use 

final_features = [
    'renewables_share_energy', 'fossil_share_energy', 
    'coal_share_energy', 'oil_share_energy', 'gas_share_energy',
    'nuclear_share_energy',
    'renewables_share_elec', 'nuclear_share_elec'
]

countries_t0 = set(sovereign_countries[sovereign_countries.year==1995].dropna(subset=final_features)['iso_code'])
countries_t1 = set(sovereign_countries[sovereign_countries.year==2010].dropna(subset=final_features + ['gdp'])['iso_code'])
countries_t2 = set(sovereign_countries[sovereign_countries.year==2015].dropna(subset=['gdp'])['iso_code'])

final_survivors = countries_t0 & countries_t1 & countries_t2
print(len(final_survivors))

77


In [12]:
sub_1995 = sovereign_countries[(sovereign_countries.year==1995) & (sovereign_countries.iso_code.isin(final_survivors))]
print(sub_1995[final_features].eq(0).sum())
print()
print((sub_1995[final_features] < 0.01).sum())  # near-zero, not just exactly 0

renewables_share_energy     6
fossil_share_energy         0
coal_share_energy           7
oil_share_energy            0
gas_share_energy            5
nuclear_share_energy       48
renewables_share_elec       6
nuclear_share_elec         48
dtype: int64

renewables_share_energy     6
fossil_share_energy         0
coal_share_energy           9
oil_share_energy            0
gas_share_energy            5
nuclear_share_energy       48
renewables_share_elec       6
nuclear_share_elec         48
dtype: int64


In [13]:
# Step 1: Creating three slices, each keeping iso_code + the relevant columns,
# renamed with a year suffix so nothing collides when merged

t0_slice = sovereign_countries[
    (sovereign_countries.year == 1995) & 
    (sovereign_countries.iso_code.isin(final_survivors))
][['iso_code'] + final_features].copy()
t0_slice = t0_slice.rename(columns={col: f"{col}_1995" for col in final_features})

t1_slice = sovereign_countries[
    (sovereign_countries.year == 2010) & 
    (sovereign_countries.iso_code.isin(final_survivors))
][['iso_code'] + final_features + ['gdp']].copy()
t1_slice = t1_slice.rename(columns={col: f"{col}_2010" for col in final_features})
t1_slice = t1_slice.rename(columns={'gdp': 'gdp_2010'})

t2_slice = sovereign_countries[
    (sovereign_countries.year == 2015) & 
    (sovereign_countries.iso_code.isin(final_survivors))
][['iso_code', 'gdp']].copy()
t2_slice = t2_slice.rename(columns={'gdp': 'gdp_2015'})

# Step 2: Merging all three into one wide table, one row per country
model_data = t0_slice.merge(t1_slice, on='iso_code', how='inner')
model_data = model_data.merge(t2_slice, on='iso_code', how='inner')

print(model_data.shape)
model_data.head(80)

(77, 19)


,iso_code,renewables_share_energy_1995,fossil_share_energy_1995,coal_share_energy_1995,oil_share_energy_1995,gas_share_energy_1995,nuclear_share_energy_1995,renewables_share_elec_1995,nuclear_share_elec_1995,renewables_share_energy_2010,fossil_share_energy_2010,coal_share_energy_2010,oil_share_energy_2010,gas_share_energy_2010,nuclear_share_energy_2010,renewables_share_elec_2010,nuclear_share_elec_2010,gdp_2010,gdp_2015
0,DZA,0.171,99.829,2.303,32.941,64.585,0.000,0.979,0.000,0.108,99.892,0.821,41.373,57.698,0.000,0.372,0.000,4.507058e+11,5.515217e+11
1,ARG,11.830,85.032,1.185,41.882,41.965,3.138,39.635,10.520,11.092,86.765,1.538,37.183,48.044,2.143,27.595,5.722,7.857852e+11,8.503566e+11
2,AUS,4.002,95.998,42.049,37.152,16.798,0.000,9.408,0.000,4.088,95.912,40.350,34.499,21.063,0.000,8.830,0.000,9.938540e+11,1.139303e+12
3,AUT,30.140,69.860,11.177,37.580,21.102,0.000,70.681,0.000,30.113,69.887,9.496,37.189,23.202,0.000,66.337,0.000,3.388322e+11,3.581083e+11
4,AZE,2.695,97.305,0.022,48.437,48.847,0.000,9.129,0.000,7.021,92.979,0.038,29.131,63.810,0.000,18.265,0.000,1.442966e+11,1.640369e+11
5,BGD,0.965,99.035,3.486,32.561,62.988,0.000,3.443,0.000,0.834,99.166,3.457,18.982,76.727,0.000,2.031,0.000,4.000878e+11,5.521047e+11
6,BLR,0.022,99.978,4.416,49.587,45.975,0.000,0.080,0.000,0.252,99.748,2.452,29.597,67.698,0.000,0.401,0.000,1.662399e+11,1.737098e+11
7,BEL,0.294,82.549,15.373,48.579,18.598,17.157,0.911,56.257,3.125,80.474,5.727,49.258,25.489,16.401,7.061,51.526,4.112617e+11,4.415515e+11
8,BRA,43.373,56.252,7.377,46.074,2.802,0.375,94.157,0.914,42.505,56.281,5.397,42.034,8.849,1.214,84.802,2.815,2.820493e+12,3.021043e+12
9,BGR,2.226,80.096,32.813,27.766,19.518,17.677,5.210,41.441,7.258,73.795,38.193,22.851,12.750,18.948,12.541,33.203,1.090452e+11,1.144986e+11


In [14]:
# Step 3: Compute absolute change for each energy feature, in a loop
for feature in final_features:
    model_data[f"{feature}_change"] = model_data[f"{feature}_2010"] - model_data[f"{feature}_1995"]

# Step 4: Compute GDP growth (percentage)
model_data['gdp_growth'] = (model_data['gdp_2015'] - model_data['gdp_2010']) / model_data['gdp_2010']

# Step 5: Median split into binary target
model_data['high_growth'] = (model_data['gdp_growth'] > model_data['gdp_growth'].median()).astype(int)

model_data.head()

,iso_code,renewables_share_energy_1995,fossil_share_energy_1995,coal_share_energy_1995,oil_share_energy_1995,gas_share_energy_1995,nuclear_share_energy_1995,renewables_share_elec_1995,nuclear_share_elec_1995,renewables_share_energy_2010,...,renewables_share_energy_change,fossil_share_energy_change,coal_share_energy_change,oil_share_energy_change,gas_share_energy_change,nuclear_share_energy_change,renewables_share_elec_change,nuclear_share_elec_change,gdp_growth,high_growth
0,DZA,0.171,99.829,2.303,32.941,64.585,0.000,0.979,0.00,0.108,...,-0.063,0.063,-1.482,8.432,-6.887,0.000,-0.607,0.000,0.223685,1
1,ARG,11.830,85.032,1.185,41.882,41.965,3.138,39.635,10.52,11.092,...,-0.738,1.733,0.353,-4.699,6.079,-0.995,-12.040,-4.798,0.082174,0
2,AUS,4.002,95.998,42.049,37.152,16.798,0.000,9.408,0.00,4.088,...,0.086,-0.086,-1.699,-2.653,4.265,0.000,-0.578,0.000,0.146349,1
3,AUT,30.140,69.860,11.177,37.580,21.102,0.000,70.681,0.00,30.113,...,-0.027,0.027,-1.681,-0.391,2.100,0.000,-4.344,0.000,0.056890,0
4,AZE,2.695,97.305,0.022,48.437,48.847,0.000,9.129,0.00,7.021,...,4.326,-4.326,0.016,-19.306,14.963,0.000,9.136,0.000,0.136804,0


In [15]:
model_data['high_growth'].value_counts()

high_growth
0    39
1    38
Name: count, dtype: int64

In [16]:
model_data.head(80)

,iso_code,renewables_share_energy_1995,fossil_share_energy_1995,coal_share_energy_1995,oil_share_energy_1995,gas_share_energy_1995,nuclear_share_energy_1995,renewables_share_elec_1995,nuclear_share_elec_1995,renewables_share_energy_2010,...,renewables_share_energy_change,fossil_share_energy_change,coal_share_energy_change,oil_share_energy_change,gas_share_energy_change,nuclear_share_energy_change,renewables_share_elec_change,nuclear_share_elec_change,gdp_growth,high_growth
0,DZA,0.171,99.829,2.303,32.941,64.585,0.000,0.979,0.000,0.108,...,-0.063,0.063,-1.482,8.432,-6.887,0.000,-0.607,0.000,0.223685,1
1,ARG,11.830,85.032,1.185,41.882,41.965,3.138,39.635,10.520,11.092,...,-0.738,1.733,0.353,-4.699,6.079,-0.995,-12.040,-4.798,0.082174,0
2,AUS,4.002,95.998,42.049,37.152,16.798,0.000,9.408,0.000,4.088,...,0.086,-0.086,-1.699,-2.653,4.265,0.000,-0.578,0.000,0.146349,1
3,AUT,30.140,69.860,11.177,37.580,21.102,0.000,70.681,0.000,30.113,...,-0.027,0.027,-1.681,-0.391,2.100,0.000,-4.344,0.000,0.056890,0
4,AZE,2.695,97.305,0.022,48.437,48.847,0.000,9.129,0.000,7.021,...,4.326,-4.326,0.016,-19.306,14.963,0.000,9.136,0.000,0.136804,0
5,BGD,0.965,99.035,3.486,32.561,62.988,0.000,3.443,0.000,0.834,...,-0.131,0.131,-0.029,-13.579,13.739,0.000,-1.412,0.000,0.379959,1
6,BLR,0.022,99.978,4.416,49.587,45.975,0.000,0.080,0.000,0.252,...,0.230,-0.230,-1.964,-19.990,21.723,0.000,0.321,0.000,0.044935,0
7,BEL,0.294,82.549,15.373,48.579,18.598,17.157,0.911,56.257,3.125,...,2.831,-2.075,-9.646,0.679,6.891,-0.756,6.150,-4.731,0.073651,0
8,BRA,43.373,56.252,7.377,46.074,2.802,0.375,94.157,0.914,42.505,...,-0.868,0.029,-1.980,-4.040,6.047,0.839,-9.355,1.901,0.071105,0
9,BGR,2.226,80.096,32.813,27.766,19.518,17.677,5.210,41.441,7.258,...,5.032,-6.301,5.380,-4.915,-6.768,1.271,7.331,-8.238,0.050010,0


In [17]:
X=model_data[["renewables_share_energy_change","fossil_share_energy_change","coal_share_energy_change","oil_share_energy_change","gas_share_energy_change","nuclear_share_energy_change","renewables_share_elec_change","nuclear_share_elec_change"]]  #Predictor 
Y=model_data["high_growth"] #Response 

print(X.shape) 
print(Y.shape) 

(77, 8)
(77,)


In [36]:
# define predictors and response
predictors = [
    "renewables_share_energy_change",
    "coal_share_energy_change",
    "nuclear_share_energy_change",
    "renewables_share_elec_change",
]
response = "high_growth"

X = model_data[predictors].copy()
y = model_data[response].copy()

print("Rows available:", X.shape[0])
print("Class balance:\n", y.value_counts(normalize=True))

# train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# scaling for Logistic Regression
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# definition of the three models
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree":       DecisionTreeClassifier(max_depth=4, random_state=42),
    "Gradient Boosting":   GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
}

results = []
fitted_models = {}

for name, model in models.items():
    if name == "Logistic Regression":
        model.fit(X_train_scaled, y_train)
        preds = model.predict(X_test_scaled)
    else:
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

    fitted_models[name] = model
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, preds),
        "Precision": precision_score(y_test, preds),
        "Recall": recall_score(y_test, preds),
        "F1": f1_score(y_test, preds),
    })

results_df = pd.DataFrame(results).set_index("Model").round(3)
print(results_df)
results_df.to_csv("model_comparison_results.csv")

# confusion matrices for all three models
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, (name, model) in zip(axes, fitted_models.items()):
    Xt = X_test_scaled if name == "Logistic Regression" else X_test
    preds = model.predict(Xt)
    cm = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=["Low growth", "High growth"]).plot(ax=ax, colorbar=False)
    ax.set_title(name)
plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.close()

# interpretability extras 
coef_df = pd.DataFrame({
    "feature": predictors,
    "coefficient": fitted_models["Logistic Regression"].coef_[0]
}).sort_values("coefficient", key=abs, ascending=False)
print("\nLogistic Regression coefficients:\n", coef_df)

importance_df = pd.DataFrame({
    "feature": predictors,
    "importance": fitted_models["Decision Tree"].feature_importances_
}).sort_values("importance", ascending=False)
print("\nDecision Tree feature importances:\n", importance_df)

plt.figure(figsize=(16, 8))
plot_tree(fitted_models["Decision Tree"], feature_names=predictors,
          class_names=["Low growth", "High growth"], filled=True, fontsize=8)
plt.savefig("decision_tree.png", dpi=150, bbox_inches="tight")
plt.close()

gb_importance_df = pd.DataFrame({
    "feature": predictors,
    "importance": fitted_models["Gradient Boosting"].feature_importances_
}).sort_values("importance", ascending=False)
print("\nGradient Boosting feature importances:\n", gb_importance_df)

Rows available: 77
Class balance:
 high_growth
0    0.506494
1    0.493506
Name: proportion, dtype: float64
                     Accuracy  Precision  Recall     F1
Model                                                  
Logistic Regression     0.688      0.667   0.750  0.706
Decision Tree           0.750      0.833   0.625  0.714
Gradient Boosting       0.750      0.750   0.750  0.750

Logistic Regression coefficients:
                           feature  coefficient
0  renewables_share_energy_change    -1.083611
2     nuclear_share_energy_change    -0.450794
3    renewables_share_elec_change    -0.447768
1        coal_share_energy_change     0.376614

Decision Tree feature importances:
                           feature  importance
0  renewables_share_energy_change    0.447118
1        coal_share_energy_change    0.319390
2     nuclear_share_energy_change    0.151954
3    renewables_share_elec_change    0.081538

Gradient Boosting feature importances:
                           feature

In [37]:
# sanity checks: train-vs-test accuracy and cross-validation
print("Train vs Test accuracy (checking overfitting):")
for name, model in fitted_models.items():
    Xtr = X_train_scaled if name == "Logistic Regression" else X_train
    Xte = X_test_scaled if name == "Logistic Regression" else X_test
    train_acc = model.score(Xtr, y_train)
    test_acc = model.score(Xte, y_test)
    print(f"{name:20s} train={train_acc:.3f}  test={test_acc:.3f}")

print("\n5-fold Cross-Validation accuracy (more reliable than single split):")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
for name, model in models.items():
    Xcv = scaler.fit_transform(X) if name == "Logistic Regression" else X
    scores = cross_val_score(model, Xcv, y, cv=cv, scoring="accuracy")
    print(f"{name:20s} mean={scores.mean():.3f}  std={scores.std():.3f}  folds={np.round(scores,2)}")

print("\nFeature correlation matrix:")
print(X.corr().round(2))

Train vs Test accuracy (checking overfitting):
Logistic Regression  train=0.770  test=0.688
Decision Tree        train=0.885  test=0.750
Gradient Boosting    train=1.000  test=0.750

5-fold Cross-Validation accuracy (more reliable than single split):
Logistic Regression  mean=0.714  std=0.135  folds=[0.62 0.81 0.93 0.6  0.6 ]
Decision Tree        mean=0.675  std=0.116  folds=[0.56 0.81 0.67 0.53 0.8 ]
Gradient Boosting    mean=0.662  std=0.094  folds=[0.56 0.75 0.73 0.53 0.73]

Feature correlation matrix:
                                renewables_share_energy_change  \
renewables_share_energy_change                            1.00   
coal_share_energy_change                                 -0.39   
nuclear_share_energy_change                              -0.06   
renewables_share_elec_change                              0.73   

                                coal_share_energy_change  \
renewables_share_energy_change                     -0.39   
coal_share_energy_change              

In [38]:
# correlation heatmap
plt.figure(figsize=(9, 7))
corr_matrix = X.corr()
sns.heatmap(corr_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            vmin=-1, vmax=1, square=True, cbar_kws={"label": "Correlation"})
plt.title("Predictor Correlation Matrix")
plt.tight_layout()
plt.savefig("correlation_heatmap.png", dpi=150)
plt.close()

# flag highly correlated pairs (|r| > 0.7)
pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        r = corr_matrix.iloc[i, j]
        if abs(r) > 0.7:
            pairs.append((corr_matrix.columns[i], corr_matrix.columns[j], round(r, 3)))

pairs_df = pd.DataFrame(pairs, columns=["Feature 1", "Feature 2", "Correlation"])
pairs_df = pairs_df.sort_values("Correlation", key=abs, ascending=False)
print("Highly correlated feature pairs (|r| > 0.7):")
print(pairs_df)

# variance Inflation Factor (VIF) — the proper multicollinearity metric
# VIF = 1        -> no correlation with other predictors
# VIF 1-5        -> moderate correlation, usually fine
# VIF > 5-10     -> problematic multicollinearity, consider dropping the feature
X_vif = add_constant(X)
vif_data = pd.DataFrame({
    "feature": X_vif.columns,
    "VIF": [variance_inflation_factor(X_vif.values, i) for i in range(X_vif.shape[1])]
})
vif_data = vif_data[vif_data["feature"] != "const"].sort_values("VIF", ascending=False)
print("\nVariance Inflation Factor (VIF) per feature:")
print(vif_data.to_string(index=False))

vif_data.to_csv("vif_results.csv", index=False)
pairs_df.to_csv("high_correlation_pairs.csv", index=False)

# suggested features to drop (VIF > 5), keeping the more interpretable/original one
high_vif = vif_data[vif_data["VIF"] > 5]["feature"].tolist()
print("\nFeatures with VIF > 5 (candidates to drop):", high_vif)

Highly correlated feature pairs (|r| > 0.7):
                        Feature 1                     Feature 2  Correlation
0  renewables_share_energy_change  renewables_share_elec_change        0.729

Variance Inflation Factor (VIF) per feature:
                       feature      VIF
renewables_share_energy_change 2.229218
  renewables_share_elec_change 2.226697
      coal_share_energy_change 1.202744
   nuclear_share_energy_change 1.040473

Features with VIF > 5 (candidates to drop): []


In [28]:
"""
Highly correlated feature pairs (|r| > 0.7):
                        Feature 1                     Feature 2  Correlation
3     nuclear_share_energy_change     nuclear_share_elec_change        0.981
0  renewables_share_energy_change    fossil_share_energy_change       -0.733
1  renewables_share_energy_change  renewables_share_elec_change        0.729
2         oil_share_energy_change       gas_share_energy_change       -0.713

Variance Inflation Factor (VIF) per feature:
                       feature          VIF
    fossil_share_energy_change 2.133545e+08
       oil_share_energy_change 1.547654e+08
renewables_share_energy_change 1.352273e+08
       gas_share_energy_change 1.205493e+08
   nuclear_share_energy_change 1.045690e+08
      coal_share_energy_change 6.322501e+07
     nuclear_share_elec_change 3.013404e+01
  renewables_share_elec_change 2.380418e+00

Features with VIF > 5 (candidates to drop): ['fossil_share_energy_change', 'oil_share_energy_change', 'renewables_share_energy_change', 'gas_share_energy_change', 'nuclear_share_energy_change', 'coal_share_energy_change', 'nuclear_share_elec_change']
"""
# after removal of the multicollinear features
"""
Highly correlated feature pairs (|r| > 0.7):
                        Feature 1                     Feature 2  Correlation
0  renewables_share_energy_change  renewables_share_elec_change        0.729

Variance Inflation Factor (VIF) per feature:
                       feature      VIF
renewables_share_energy_change 2.229218
  renewables_share_elec_change 2.226697
      coal_share_energy_change 1.202744
   nuclear_share_energy_change 1.040473

Features with VIF > 5 (candidates to drop): []


"""


SyntaxError: incomplete input (3951088659.py, line 1)

In [39]:
# regularization to reduce overfitting in tree-based models
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

regularized_models = {
    "Logistic Regression (C=0.5)": LogisticRegression(C=0.5, max_iter=1000, random_state=42),
    "Decision Tree (depth=2, leaf=8)": DecisionTreeClassifier(max_depth=2, min_samples_leaf=8, random_state=42),
    "Gradient Boosting (depth=2, n=50, leaf=8)": GradientBoostingClassifier(
        n_estimators=50, max_depth=2, min_samples_leaf=8, learning_rate=0.05, random_state=42),
}

print("Regularized models — 5-fold CV accuracy:")
for name, model in regularized_models.items():
    Xcv = scaler.fit_transform(X) if "Logistic" in name else X
    scores = cross_val_score(model, Xcv, y, cv=cv, scoring="accuracy")
    print(f"{name:40s} mean={scores.mean():.3f}  std={scores.std():.3f}")

print("\nTrain vs CV gap check (refit on full train/test split):")
for name, model in regularized_models.items():
    Xtr = X_train_scaled if "Logistic" in name else X_train
    model.fit(Xtr, y_train)
    train_acc = model.score(Xtr, y_train)
    Xcv = scaler.fit_transform(X) if "Logistic" in name else X
    cv_acc = cross_val_score(model, Xcv, y, cv=cv, scoring="accuracy").mean()
    print(f"{name:40s} train={train_acc:.3f}  cv={cv_acc:.3f}  gap={train_acc-cv_acc:.3f}")

Regularized models — 5-fold CV accuracy:
Logistic Regression (C=0.5)              mean=0.728  std=0.127
Decision Tree (depth=2, leaf=8)          mean=0.713  std=0.055
Gradient Boosting (depth=2, n=50, leaf=8) mean=0.715  std=0.062

Train vs CV gap check (refit on full train/test split):
Logistic Regression (C=0.5)              train=0.787  cv=0.728  gap=0.059
Decision Tree (depth=2, leaf=8)          train=0.787  cv=0.713  gap=0.074
Gradient Boosting (depth=2, n=50, leaf=8) train=0.902  cv=0.715  gap=0.187
